# Richter 2020 cohort overview

Descriptive figures comparing the de novo variants of CHD probands and controls:
DNV burden, genomic and coding annotation, conservation, ENCODE4 cCREs and overlap
with cardioid ATAC peaks and ABC candidate enhancers.

**Input:** `VCF_all_Richter_vars.csv` (written by `richter_variant_dataset.ipynb`) and the
pseudobulk ATAC peaks (`atac_peak_dir`). Saving the figures (to `figures_dir/richter_cohort`)
is commented out.

In [ ]:
import glob
import os

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu

from cardiomics_s2f.config import load_paths
from cardiomics_s2f.io.variant_annotation import (
    CONSEQUENCE_RANK,
    IMPACT,
    celltype_from_peak_file,
    flag_variants_in_peaks,
    read_narrowpeak,
)
from cardiomics_s2f.vep.cohort_plotting import (
    CRE_ORDER,
    format_p_value,
    label_cardioid_enhancer_state,
    plot_richter_cohort_overview,
    plot_richter_cohort_qc,
)

In [ ]:
paths = load_paths()
Richter_all_vcf = pd.read_csv(paths.variant_data_dir / "VCF_all_Richter_vars.csv")
FIGURE_DIR = paths.figures_dir / "richter_cohort"

# Publication style for all figures
sns.set_theme(style="ticks", context="talk", font_scale=0.85)
plt.rcParams['font.family'] = 'sans-serif'

## Overview figures

Variants inside and outside the cardioid ATAC peaks of each cell type (panels J and K
of the overview figure).

In [ ]:
peak_files = glob.glob(os.path.join(paths.atac_peak_dir, "*_peaks.narrowPeak"))
cell_types = {celltype_from_peak_file(p): p for p in peak_files}
print("Cell types found:", list(cell_types.keys()))

# flag_variants_in_peaks expects Chromosome, Start and End columns
peak_input = Richter_all_vcf.rename(columns={"CHROM": "Chromosome"})

CA_variants = {}
for ct, pf in cell_types.items():
    peaks = read_narrowpeak(pf)
    mask  = flag_variants_in_peaks(peak_input, peaks)
    ct_results = {}
    for cohort, grp in Richter_all_vcf.groupby("Cohort", observed=False):
        grp_mask = mask[grp.index]
        ct_results[cohort] = {
            "in":  grp[grp_mask],
            "out": grp[~grp_mask],
        }
    CA_variants[ct] = ct_results
    print(f"  {ct}: done")

In [ ]:
Richter_CHD_vars = Richter_all_vcf[Richter_all_vcf.is_CHD]
Richter_control_vars = Richter_all_vcf[~Richter_all_vcf.is_CHD]

In [ ]:
plot_richter_cohort_overview(Richter_CHD_vars, Richter_control_vars, CA_results_dict=CA_variants,
                             celltype_filter = ['Cardiomyocytes', 'Fibroblasts'],
                             # save_fig=True, filepath=FIGURE_DIR / 'Richter_descriptive_panel.png',
                            )

In [ ]:
plot_richter_cohort_qc(Richter_CHD_vars.rename(columns={"REF":"Ref","ALT":"Alt"}),
                       Richter_control_vars.rename(columns={"REF":"Ref","ALT":"Alt"}),
                       # save_fig=True, filepath=FIGURE_DIR / 'Richter_descriptive_panel_supp.png',
                      )

## Individual figures

In [ ]:
#use true fonts for pdf export
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none' 

# Set Theme and explicitly request Helvetica
sns.set_theme(style="ticks", context="talk", font_scale=0.85)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Helvetica', 'Arial', 'sans-serif'] # Fallback to Arial if Helvetica isn't found

color_chd = "#E63946"
color_ctrl = "#2A9D8F"
element_width = 0.6

df_combined = Richter_all_vcf
celltype_filter = ['Cardiomyocytes', 'Cardiac_progenitors', 'Cardiac_mesoderm', 'Fibroblasts']

In [ ]:
# 1. Clean up missing data in the reference column
df_combined['cCRE_class'] = df_combined['cCRE_class'].fillna('No CRE')

# 2. Create a composite "Measured State" based on your cardioid data

df_combined['Measured_State'] = df_combined.apply(label_cardioid_enhancer_state, axis=1)

# 3. Generate the Cross-Tabulation
# normalize='index' ensures each row (cCRE class) sums to 100%
cross_tab_pct = pd.crosstab(
    df_combined['cCRE_class'], 
    df_combined['Measured_State'], 
    normalize='index'
) * 100

# Also grab raw counts so we can annotate the total variants per row
cross_tab_counts = pd.crosstab(df_combined['cCRE_class'], df_combined['Measured_State'])
row_totals = cross_tab_counts.sum(axis=1)

# Sort the index to match your standard CRE_ORDER
valid_order = [c for c in CRE_ORDER + ['No CRE'] if c in cross_tab_pct.index]
cross_tab_pct = cross_tab_pct.reindex(valid_order)
row_totals = row_totals.reindex(valid_order)

# Rename index to include the raw N for context
cross_tab_pct.index = [f"{idx} (n={row_totals[idx]})" for idx in cross_tab_pct.index]

# --- NEW: Reorder columns to put "Not Active in Cardioid" on the far right ---
cols = list(cross_tab_pct.columns)
if "Not Active in Cardioid" in cols:
    cols.remove("Not Active in Cardioid")
    cols.sort() # Alphabetize the remaining columns to neatly group Distal/Proximal
    cols.append("Not Active in Cardioid")
    cross_tab_pct = cross_tab_pct[cols] # Reapply the new sorted order

# 4. Plot the Heatmap
fig, ax = plt.subplots(figsize=(7, 5))

sns.heatmap(
    cross_tab_pct, 
    annot=True,          
    fmt=".1f",           
    cmap="viridis",              # UPDATED: Viridis color scheme
    vmin=0, vmax=100,            # UPDATED: Lock legend from 0 to 100%
    square=True,                 # UPDATED: Force cells to be perfectly square
    linewidths=0.5,      
    linecolor='black',           
    annot_kws={"fontsize": 8, "fontweight": "bold"},   # UPDATED: Smaller font size inside the cells
    cbar_kws={'label': 'Variants of cCRE Class (%)', 'shrink': 0.7},
    ax=ax
)

# --- NEW: Grab the colorbar object to reduce its text sizes ---
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=8)  # Shrink the numbers (0, 20, 40...)
cbar.set_label('Variants of cCRE Class (%)', size=9) # Shrink the title

# UPDATED: Draw a solid black border around the entire heatmap
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(1.5)

# UPDATED: Remove ticks (little lines) and make tick labels smaller
ax.tick_params(left=False, bottom=False, labelsize=9)

# Rotate labels for readability
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

# UPDATED: Make X and Y axis titles bold and slightly smaller
ax.set_ylabel("ENCODE4 cCRE", fontweight='bold', fontsize=10)
ax.set_xlabel("Cardioid Measurement", fontweight='bold', fontsize=10)

ax.set_title("Cardioid enhancer and ENCODE4 cCRE Concordance\n(Variants in elements, CHD and control combined)", 
             loc='center', fontweight='bold', pad=15, fontsize=11)

plt.tight_layout()
# plt.savefig(FIGURE_DIR / 'Cardioid enhancer and ENCODE4 cCRE Concordance.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
fig, ax_a = plt.subplots(figsize=(7, 5)) 

total_chd, n_noncausal, n_control = 763, 749, 1611
n_causal = total_chd - n_noncausal + 3 #the three we filtered out

ax_a.bar(['CHD removed\nCausal Found', 'CHD Retained\nNon-Coding', 'Control Retained\nNon-Coding'], 
         [np.log10(n_causal), np.log10(n_noncausal), np.log10(n_control)], color=['#333333', color_chd, color_ctrl], width=element_width, edgecolor="black")
ax_a.set_ylabel('Log10 Number of Probands')
ax_a.set_title('CHD Proband Filtering', loc='left', fontweight='bold')
ax_a.spines[['top', 'right']].set_visible(False)

for i, v in enumerate([n_causal, n_noncausal, n_control]):
    ax_a.text(i, np.log10(v) + 0.05, str(v), ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout() # Ensures labels don't get cut off

# plt.savefig(FIGURE_DIR / 'CHD_Proband_Filtering.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
fig, ax_g = plt.subplots(figsize=(7, 5)) 

dnvs_per_proband = df_combined.groupby(['Cohort', 'Blinded ID']).size().reset_index(name='DNV_Count')

sns.histplot(data=dnvs_per_proband, x='DNV_Count', hue='Cohort', element='step', stat='density',
             common_norm=False, palette=[color_chd, color_ctrl], ax=ax_g, bins=25)

ax_g.set_title('DNV Burden per Proband', loc='left', fontweight='bold')
ax_g.set_ylabel('Density')
ax_g.set_xlabel('DNVs per Proband')
ax_g.spines[['top', 'right']].set_visible(False)
ax_g.get_legend().remove()

counts_chd = dnvs_per_proband[dnvs_per_proband['Cohort'] == 'CHD']['DNV_Count']
counts_ctrl = dnvs_per_proband[dnvs_per_proband['Cohort'] == 'Control']['DNV_Count']
if len(counts_chd) > 0 and len(counts_ctrl) > 0:
    _, p_val_burden = mannwhitneyu(counts_chd, counts_ctrl, alternative='two-sided')
    ax_g.text(0.95, 0.95, format_p_value(p_val_burden), transform=ax_g.transAxes, 
              ha='right', va='top', fontweight='bold', fontsize=10)

plt.tight_layout()

# plt.savefig(FIGURE_DIR / 'DNV_Burden_per_Proband.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
fig, ax_e = plt.subplots(figsize=(6, 8)) 

df_combined["gtf_feature"] = df_combined["gtf_feature"].fillna("Intergenic")
df_combined = df_combined[df_combined["gtf_feature"] != "gene"] #drop the one case of gene 

# 1. Define the explicit order and color mapping
feature_order = [
    'start_codon', 'stop_codon', 'CDS', 'exon', 
    'UTR', 'intron', 'Intergenic'
]

feature_colors = {
    # Greys (Pure Black -> Medium Grey)
    'start_codon':  '#999999',
    'stop_codon':   '#666666',
    'CDS':          '#333333',
    'exon':         '#000000',
    # Reds (Dark Red -> Light Red)
    'UTR':          '#FFB2B2',
    'intron':       '#CC0000',
 #   'gene':         '#CC0000',
    'Intergenic':    '#8B0000',
}

# Calculate unstacked counts
gtf_counts = df_combined.groupby(['Cohort', 'gtf_feature']).size().unstack(fill_value=0)

# 2. Enforce the order (only keep categories that actually exist in your data)
valid_order = [col for col in feature_order if col in gtf_counts.columns]
gtf_counts = gtf_counts.reindex(columns=valid_order, fill_value=0)

gtf_pct = gtf_counts.div(gtf_counts.sum(axis=1), axis=0) * 100

# 3. Build the color palette based on the valid order
gtf_colors_mapped = [feature_colors[col] for col in valid_order]

# Plot stacked bar using our custom colors
gtf_pct.plot(kind='bar', stacked=True, color=gtf_colors_mapped, ax=ax_e, 
             edgecolor="black", width=element_width, legend=False)

ax_e.set_title('Variant GTF Annotations', loc='left', fontweight='bold')
ax_e.set_ylabel('Proportion of DNVs (%)')
ax_e.set_xlabel('')
ax_e.set_xticklabels(ax_e.get_xticklabels(), rotation=0)
ax_e.spines[['top', 'right']].set_visible(False)

# Custom Legend with integrated counts placed below the plot
legend_labels_e = []
for i, col in enumerate(gtf_pct.columns):
    chd_n = gtf_counts.loc['CHD', col] if 'CHD' in gtf_counts.index else 0
    ctrl_n = gtf_counts.loc['Control', col] if 'Control' in gtf_counts.index else 0
    legend_labels_e.append(f"{col} (n={int(chd_n)} vs {int(ctrl_n)})")

handles_e = [plt.Rectangle((0,0),1,1, color=gtf_colors_mapped[i], ec="black") for i in range(len(gtf_pct.columns))]

# Depending on how long this list is, you might want to switch ncol=2 to save vertical space
ax_e.legend(handles_e, legend_labels_e, title='GTF Feature (CHD vs Ctrl)', 
            bbox_to_anchor=(0.5, -0.15), loc='upper center', frameon=False, fontsize=9, ncol=2)

plt.tight_layout()

# plt.savefig(FIGURE_DIR / 'GTF_annotation_per_variant.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
fig, ax_j = plt.subplots(figsize=(8, 5)) 

df_coding = df_combined.dropna(subset=['coding_effect'])

if not df_coding.empty:
    # 1. Calculate unstacked counts
    effect_counts = df_coding.groupby(['Cohort', 'coding_effect']).size().unstack(fill_value=0)

    # 2. Identify present effects and sort them strictly by CONSEQUENCE_RANK
    present_effects = list(effect_counts.columns)
    # Default to 999 if an effect isn't in the dict so it goes to the bottom safely
    present_effects = sorted(present_effects, key=lambda x: CONSEQUENCE_RANK.get(x, 999))

    # Reorder the dataframe columns to match the rank
    effect_counts = effect_counts[present_effects]
    effect_pct = effect_counts.div(effect_counts.sum(axis=1), axis=0) * 100

    # 3. Dynamically assign colors based on IMPACT
    # Count how many of each impact level are present in the data
    impact_counts = {'HIGH': 0, 'MODERATE': 0, 'LOW': 0, 'MODIFIER': 0}
    for eff in present_effects:
        imp = IMPACT.get(eff, 'MODIFIER') # Default unknown to MODIFIER
        impact_counts[imp] += 1

    # Generate gradients (using _r to go from dark to light)
    # We request N+1 colors and drop the last one [:-1] so we don't end up with pure white
    palettes = {
        'HIGH': sns.color_palette("Greys_r", impact_counts['HIGH'] + 1)[:-1] if impact_counts['HIGH'] else [],
        'MODERATE': sns.color_palette("Reds_r", impact_counts['MODERATE'] + 1)[:-1] if impact_counts['MODERATE'] else [],
        'LOW': sns.color_palette("Greens", impact_counts['LOW'] + 1)[:-1] if impact_counts['LOW'] else [],
        'MODIFIER': sns.color_palette("Blues_r", impact_counts['MODIFIER'] + 1)[:-1] if impact_counts['MODIFIER'] else []
    }
            
    # Map the generated colors to the sorted effects
    color_map = {}
    pal_idx = {'HIGH': 0, 'MODERATE': 0, 'LOW': 0, 'MODIFIER': 0}

    for eff in present_effects:
        imp = IMPACT.get(eff, 'MODIFIER')
        color_map[eff] = palettes[imp][pal_idx[imp]]
        pal_idx[imp] += 1

    # Create the final ordered list of colors for plotting
    effect_colors = [color_map[eff] for eff in present_effects]

    # 4. Plot stacked bar
    effect_pct.plot(kind='bar', stacked=True, color=effect_colors, ax=ax_j, 
                    edgecolor="black", width=element_width, legend=False)

    ax_j.set_title('Coding Effects', loc='left', fontweight='bold')
    ax_j.set_ylabel('Proportion of Coding DNVs (%)')
    ax_j.set_xlabel('')
    ax_j.set_xticklabels(ax_j.get_xticklabels(), rotation=0)
    ax_j.spines[['top', 'right']].set_visible(False)

    # 5. Custom Legend with integrated counts placed below the plot
    legend_labels_j = []
    for i, col in enumerate(present_effects):
        chd_n = effect_counts.loc['CHD', col] if 'CHD' in effect_counts.index else 0
        ctrl_n = effect_counts.loc['Control', col] if 'Control' in effect_counts.index else 0
        legend_labels_j.append(f"{col} (n={int(chd_n)} vs {int(ctrl_n)})")

    handles_j = [plt.Rectangle((0,0),1,1, color=effect_colors[i], ec="black") for i in range(len(present_effects))]

    # Depending on how many unique coding effects you have, you might want to set ncol=2 here if the list gets too long
    ax_j.legend(handles_j, legend_labels_j, title='', 
                bbox_to_anchor=(1.1, 1), loc='upper left', frameon=False, fontsize=9, ncol=1)

    # 6. Calculate and place P-value
    contingency_table_j = pd.crosstab(df_coding['Cohort'], df_coding['coding_effect'])
    if not contingency_table_j.empty:
        try: 
            _, p_val_j, _, _ = chi2_contingency(contingency_table_j)
            ax_j.text(0.7, 1, format_p_value(p_val_j), transform=ax_j.transAxes, 
                      ha='right', va='top', fontweight='bold', fontsize=10)
        except ValueError:
            pass 

plt.tight_layout()

# plt.savefig(FIGURE_DIR / 'ENSEMBL_variant_effects_in_coding_regions.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
fig, ax_n = plt.subplots(figsize=(7, 5)) 

# 1. Define explicit colors (Light versions instead of just alpha)
color_dict = {
    'Proximal_ac':   '#F4A261',  # Original Proximal
    'Proximal_nac':  '#FCD5B5',  # Lighter Proximal
    'Distal_ac':     '#2A9D8F',  # Original Distal
    'Distal_nac':    '#82C2BA'   # Lighter Distal
}

# 2. Extract and explode cell types for variants in enhancers
df_enh = df_combined[df_combined['in_cardioid_enhancer'] == True].copy()

# Split the comma-separated string into a list and explode so each row is one cell type match
df_enh['celltype_list'] = df_enh['celltypes_in_enhancer'].astype(str).str.split(', ')
df_exploded = df_enh.explode('celltype_list')

# Clean up any potential whitespace and get unique cell types
df_exploded['celltype_list'] = df_exploded['celltype_list'].str.strip()
all_enh_cts = sorted([ct for ct in df_exploded['celltype_list'].unique() if ct != 'nan' and ct != 'None'])

# --- FILTER CELLTYPES FOR PANEL N ---
if celltype_filter is not None:
    # Keep only the items that exist in our data, but strictly follow the order of celltype_filter
    unique_enh_cts = [ct for ct in celltype_filter if ct in all_enh_cts]
else:
    unique_enh_cts = all_enh_cts

# Ensure cohort_totals exists (fallback if not defined earlier in your script)
if 'cohort_totals' not in locals():
    cohort_totals = df_combined['Cohort'].value_counts().to_dict()

# 3. Plotting Setup
xtick_pos_enh, xtick_label_enh = [], []
bar_order_enh = ["CHD", "Control"]

x_step = 1.0
bar_w = 0.35
max_pct_total = 0 

for gi, ct in enumerate(unique_enh_cts):
    gc = gi * x_step
    xtick_pos_enh.append(gc)
    xtick_label_enh.append(ct)

    # Subset to the current cell type
    df_ct = df_exploded[df_exploded['celltype_list'] == ct]

    for bi, cohort in enumerate(bar_order_enh):
        x = gc + (bi - 0.5) * bar_w
        df_cohort = df_ct[df_ct['Cohort'] == cohort]

        n_in = len(df_cohort)
        n_total = cohort_totals.get(cohort, 1) # Prevent division by zero

        if n_in > 0:
            # Safely standardize text
            reg_pos = df_cohort['enhancer_regulatory_position'].astype(str).str.lower()
            is_ac = df_cohort['is_cardioid_H3K27ac'] == True

            # Calculate counts for the 4 categories
            n_prox_ac  = ((reg_pos == 'proximal') & is_ac).sum()
            n_prox_nac = ((reg_pos == 'proximal') & ~is_ac).sum()
            n_dist_ac  = ((reg_pos == 'distal') & is_ac).sum()
            n_dist_nac = ((reg_pos == 'distal') & ~is_ac).sum()

            # Convert to percentages of the FULL cohort
            pct_prox_ac  = 100 * n_prox_ac / n_total
            pct_prox_nac = 100 * n_prox_nac / n_total
            pct_dist_ac  = 100 * n_dist_ac / n_total
            pct_dist_nac = 100 * n_dist_nac / n_total

            pct_total = pct_prox_ac + pct_prox_nac + pct_dist_ac + pct_dist_nac

            if pct_total > max_pct_total:
                max_pct_total = pct_total

            # Stack the bars: bottom to top
            current_bottom = 0

            if pct_prox_ac > 0:
                ax_n.bar(x, pct_prox_ac, bar_w, bottom=current_bottom, color=color_dict['Proximal_ac'], edgecolor="black", linewidth=0.5)
                current_bottom += pct_prox_ac

            if pct_prox_nac > 0:
                ax_n.bar(x, pct_prox_nac, bar_w, bottom=current_bottom, color=color_dict['Proximal_nac'], edgecolor="black", linewidth=0.5)
                current_bottom += pct_prox_nac

            if pct_dist_ac > 0:
                ax_n.bar(x, pct_dist_ac, bar_w, bottom=current_bottom, color=color_dict['Distal_ac'], edgecolor="black", linewidth=0.5)
                current_bottom += pct_dist_ac

            if pct_dist_nac > 0:
                ax_n.bar(x, pct_dist_nac, bar_w, bottom=current_bottom, color=color_dict['Distal_nac'], edgecolor="black", linewidth=0.5)
                current_bottom += pct_dist_nac

            text_col = "#E63946" if cohort == "CHD" else "#2A9D8F"
            prefix = "CHD" if cohort == "CHD" else "Ctrl"

            ax_n.text(x, pct_total + (max_pct_total * 0.02 if max_pct_total > 0 else 0.1), f"{prefix} ({n_in})",
                      ha="center", va="bottom", rotation=90, fontsize=8, color=text_col, fontweight="bold")

# 4. Axis formatting
ax_n.set_xticks(xtick_pos_enh)
ax_n.set_xticklabels(xtick_label_enh, rotation=25, ha="right", fontsize=10)
ax_n.set_ylabel("Variants in Enhancers\n(% of full cohort)")
ax_n.set_title("Variants in candidate enhancers", loc='left', fontweight="bold")

if max_pct_total > 0:
    ax_n.set_ylim(0, max_pct_total * 1.35)
ax_n.spines[["top", "right"]].set_visible(False)

# 5. Legend
leg_enh_handles = [
    mpatches.Patch(facecolor=color_dict['Proximal_ac'], edgecolor='black', linewidth=0.5, label="Proximal (H3K27ac+)"),
    mpatches.Patch(facecolor=color_dict['Proximal_nac'], edgecolor='black', linewidth=0.5, label="Proximal (H3K27ac-)"),
    mpatches.Patch(facecolor=color_dict['Distal_ac'], edgecolor='black', linewidth=0.5, label="Distal (H3K27ac+)"),
    mpatches.Patch(facecolor=color_dict['Distal_nac'], edgecolor='black', linewidth=0.5, label="Distal (H3K27ac-)")
]
ax_n.legend(handles=leg_enh_handles, loc="upper left", bbox_to_anchor=(1.02, 1), 
            frameon=False, fontsize=10, title="Enhancer Annotation", title_fontsize=11)

plt.tight_layout()

# plt.savefig(FIGURE_DIR / 'Variants_in_candidate_enhancers.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.show()